# Digitalify: from QONNX to bit-exact VHDL

This notebook walks through the full pipeline:

1. get a quantised network in **QONNX** format
2. import it into digitalify's integer IR
3. **verify** that the integer model matches the QONNX model
4. generate structural VHDL
5. **simulate** the VHDL with GHDL against the integer model

No PyTorch is needed: digitalify only depends on `numpy`, `onnx` and `qonnx`.

## 1. A QONNX model

In a real project the model comes from your training framework, for example Brevitas:

```python
from brevitas.export import export_qonnx
export_qonnx(brevitas_model, torch.randn(1, 16), "model.onnx")
```

or from QKeras via `qonnx.converters.from_keras`. Here we build one directly with
`digitalify.testing.make_mlp`, which emits the same structure Brevitas does:
`Quant → MatMul(Quant(W)) → Add(bias) → Relu → Quant → ...`

The network is a 16 → 8 → 4 MLP with **4-bit weights**, **6-bit unsigned hidden
activations**, INT8 input and INT8 output.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import onnx
import digitalify
from digitalify.testing import make_mlp

model = make_mlp([16, 8, 4], weight_bits=4, act_bits=6, output_bits=8, seed=1)
onnx.save(model, 'mlp_qonnx.onnx')      # digitalify accepts a path, ModelProto or ModelWrapper

for node in model.graph.node:
    print(f'{node.op_type:<8} {list(node.input)} -> {list(node.output)}')

## 2. Import into the integer IR

`from_qonnx` cleans the model up with `qonnx` (shape inference, constant folding,
batch size 1) and fuses each `MatMul → Add → Relu → Quant` chain into one `linear`
node with integer weights, an integer bias and a fixed-point requantisation
`q = clip(round(acc · MULT / 2^SHIFT))`.

In [ ]:
ir = digitalify.from_qonnx('mlp_qonnx.onnx')
print(ir.summary())

layer = ir.nodes[0]
print()
print('weights (first row):', layer.attrs['weight'][0])
print('bias               :', layer.attrs['bias'])
print('requant MULT       :', layer.attrs['requant']['multiplier'])
print('requant SHIFT      :', layer.attrs['requant']['shift'])

## 3. Verify against QONNX execution

`verify` feeds the same integer inputs to the bit-exact integer model and to
`qonnx.core.onnx_exec` running the original model, and compares the outputs in LSBs.
With power-of-two scales they must match exactly.

In [ ]:
check = digitalify.verify('mlp_qonnx.onnx', n_vectors=256)
print(check.summary())

print()
print('input scale :', ir.input_scale)
print('output scale:', ir.output_scale)
for q_out in check.reference[3:6]:
    print('q_out =', q_out, '  real =', q_out * ir.output_scale)

## 4. Generate VHDL

One file per layer plus a structural top level. Entity names are prefixed with
the network name.

In [ ]:
OUTPUT_DIR = 'generated_vhdl'
vhdl_files = digitalify.convert('mlp_qonnx.onnx', output_dir=OUTPUT_DIR)

for fname, src in vhdl_files.items():
    print(f'{fname:<20} {src.count(chr(10)):5d} lines')

In [ ]:
print(vhdl_files['mlp_top.vhd'])

In [ ]:
# The linear component: constants, combinational MAC, registered requantisation
src = vhdl_files['mlp_linear_0.vhd']
start = src.index('    -- Combinational')
print(src[:1200])
print('    ...\n')
print(src[start:])

## 5. Error handling

Anything without a hardware mapping is rejected at import time, before any VHDL
is written: `UnsupportedOperatorError` for unknown operators or graph shapes,
`QONNXImportError` for quantisation schemes integer hardware cannot express.

In [ ]:
from onnx import TensorProto, helper, numpy_helper
from digitalify import UnsupportedOperatorError, QONNXImportError

D = 'qonnx.custom_op.general'
inits = [numpy_helper.from_array(np.array(v, dtype=np.float32), n)
         for n, v in [('s', 0.25), ('z', 0.0), ('b', 8.0)]]
nodes = [
    helper.make_node('Quant', ['x', 's', 'z', 'b'], ['xq'], domain=D, signed=1, narrow=0),
    helper.make_node('Sigmoid', ['xq'], ['y']),
]
graph = helper.make_graph(nodes, 'unsupported',
    [helper.make_tensor_value_info('x', TensorProto.FLOAT, [1, 4])],
    [helper.make_tensor_value_info('y', TensorProto.FLOAT, [1, 4])], initializer=inits)
bad = helper.make_model(graph, opset_imports=[helper.make_opsetid('', 13), helper.make_opsetid(D, 1)])

try:
    digitalify.convert(bad)
except (UnsupportedOperatorError, QONNXImportError) as e:
    print(f'{type(e).__name__}: {e}')

## 6. Simulate with GHDL

`simulate` generates a self-checking testbench whose expected outputs come from
the integer model, and runs it through GHDL. Together with step 3 this shows
that the VHDL computes exactly what the QONNX model computes.

GHDL must be on `PATH` or pointed to by the `DIGITALIFY_GHDL` environment variable
(Windows: unzip a release from https://github.com/ghdl/ghdl/releases;
macOS: `brew install ghdl`; Ubuntu: `sudo apt install ghdl`).

In [ ]:
import re
from digitalify.testing import find_ghdl

try:
    find_ghdl()
    result = digitalify.simulate('mlp_qonnx.onnx', n_vectors=16)
    print(result.summary())
    for line in result.log.splitlines():
        m = re.search(r'(Test \d+: \w+.*|TB SUMMARY.*)', line)
        if m:
            print('  ' + m.group(1))
except FileNotFoundError as e:
    print(e)

## Next steps

* Synthesise the files in `generated_vhdl/` with Vivado — locally or through the
  `vivado_api` service in this repository (top entity: `mlp_top`).
* Use your own model: `digitalify.convert('model.onnx', output_dir=...)`.
* To support a new operator, map its QONNX pattern in
  `digitalify/frontend/qonnx_importer.py`, add a `VHDLOperator` in
  `digitalify/operators/`, register it, and add its bit-exact model to
  `digitalify/testing/reference.py`.